In [ ]:
pip install rdflib pandas openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 4.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

# -----------------------------
# Define measurements using exact Excel column names
# -----------------------------
MEASUREMENTS = {
    "Sleep": "Sleep",
    "DeepSleep": "DeepSleep",
    "HeartRate": "HeartRate",
    "BloodOxygen": "BloodOxygen",
    "Calories": "Calories",
    "SkinTemp": "SkinTemp",
    "BodyFat": "BodyFat",
    "MuscleMass": "MuscleMass",
}

# -----------------------------
# Load data
# -----------------------------
df = pd.read_excel("personal_health_data2.xlsx")
HEALTH_SCORE_COLUMN = "Health_Score"

# -----------------------------
# Create directed graph
# -----------------------------
G = nx.DiGraph()

# -----------------------------
# Add patient nodes with features
# -----------------------------
for idx, row in df.iterrows():
    user_id = str(row["User_ID"])
    patient = f"Patient{user_id}"

    G.add_node(
        patient,
        type="Patient",
        age=None if pd.isna(row.get("Age")) else int(row["Age"]),
        gender=row.get("Gender"),
        weight=None if pd.isna(row.get("Weight")) else float(row["Weight"]),
        height=None if pd.isna(row.get("Height")) else float(row["Height"]),
        medical_conditions=row.get("Medical_Conditions"),
        smoker=bool(row["Smoker"]) if not pd.isna(row.get("Smoker")) else None,
        alcohol_consumption=row.get("Alcohol_Consumption")
    )

# -----------------------------
# Add Normal/Abnormal nodes for each measurement and connect patients
# -----------------------------
for measurement_col, measurement_name in MEASUREMENTS.items():
    # Create Normal and Abnormal nodes
    normal_node = f"Normal {measurement_name}"
    abnormal_node = f"Abnormal {measurement_name}"

    if not G.has_node(normal_node):
        G.add_node(normal_node, type="Measurement", status="Normal")
    if not G.has_node(abnormal_node):
        G.add_node(abnormal_node, type="Measurement", status="Abnormal")

    # Connect patients based on 0/1 in Excel
    for idx, row in df.iterrows():
        patient = f"Patient{row['User_ID']}"
        value = row.get(measurement_col)
        if pd.isna(value):
            continue  # skip missing values

        if value == 0:
            G.add_edge(patient, normal_node)
        elif value == 1:
            G.add_edge(patient, abnormal_node)

# -----------------------------
# Add HealthScore nodes and connect to patients
# -----------------------------
for _, row in df.iterrows():
    patient_id = str(row["User_ID"])
    patient_node = f"Patient{patient_id}"

    # Skip if HealthScore is missing
    if pd.isna(row.get(HEALTH_SCORE_COLUMN)):
        continue

    healthscore_value = row[HEALTH_SCORE_COLUMN]

    # Create HealthScore node
    healthscore_node = f"HealthScore_{patient_id}"
    G.add_node(
        healthscore_node,
        type="HealthScore",
        value=healthscore_value
    )

    # Add edge from patient to HealthScore
    G.add_edge(patient_node, healthscore_node)

# -----------------------------
# Optional: visualize the graph
# -----------------------------
# nx.draw(G, with_labels=True)
# plt.show()


In [ ]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 12.6 MB/s eta 0:00:00


In [ ]:
import torch
from torch_geometric.data import Data
import networkx as nx
import random

# -------------------------------
# Helper preprocessing functions
# -------------------------------
def safe_float(value, default=0.0):
    try:
        return float(value)
    except (TypeError, ValueError):
        return default

def encode_smoker(value):
    if value in [1, "1", True, "Yes", "yes", "Y"]:
        return 1
    return 0

def encode_alcohol(value):
    alcohol_map = {"None": 0, "Low": 1, "Moderate": 2, "High": 3}
    return alcohol_map.get(value, 0)

# -------------------------------
# 1️⃣ Create train graph
# -------------------------------
train_node_list = list(G.nodes())  # all nodes including HealthScore
train_node_to_idx = {node: i for i, node in enumerate(train_node_list)}

# Edges
train_edges = [(train_node_to_idx[u], train_node_to_idx[v]) for u, v in G.edges()]
train_edge_index = torch.tensor(train_edges, dtype=torch.long).t().contiguous()

# Node features and labels
train_node_features = []
train_y = []

for n in train_node_list:
    data_n = G.nodes[n]
    node_type = data_n['type']
    features = [0.0]*10  # 10-dim feature vector

    if node_type == "Patient":
        features[0] = 1  # patient indicator
        features[1:4] = [
            safe_float(data_n.get('age')),
            safe_float(data_n.get('weight')),
            safe_float(data_n.get('height'))
        ]
        features[4] = encode_smoker(data_n.get('smoker'))
        features[5] = encode_alcohol(data_n.get('alcohol_consumption'))
        train_y.append(safe_float(data_n.get("Health_Score", 0)))

    elif node_type == "Measurement":
        # Encode Normal vs Abnormal from the node name
        if n.startswith("Normal"):
            features[8] = 1  # Normal measurement
            features[9] = 0
        elif n.startswith("Abnormal"):
            features[8] = 0
            features[9] = 1
        else:
            features[8] = 0
            features[9] = 0

        train_y.append(-1)  # no target

    elif node_type == "HealthScore":
        features[7] = 1  # HealthScore node type
        train_y.append(data_n.get('value', 0))

    train_node_features.append(features)

train_x = torch.tensor(train_node_features, dtype=torch.float)
train_y = torch.tensor(train_y, dtype=torch.float)
train_data = Data(x=train_x, edge_index=train_edge_index, y=train_y)

# -------------------------------
# 2️⃣ Create train/val/test masks (only Patient + HealthScore nodes)
# -------------------------------
num_nodes = len(train_node_list)
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask = torch.zeros(num_nodes, dtype=torch.bool)
test_mask = torch.zeros(num_nodes, dtype=torch.bool)

patient_and_hs_indices = [
    i for i, n in enumerate(train_node_list)
    if G.nodes[n]['type'] in ["Patient", "HealthScore"]
]

random.shuffle(patient_and_hs_indices)
n_total = len(patient_and_hs_indices)
train_split = int(0.7 * n_total)
val_split = int(0.85 * n_total)

train_mask[patient_and_hs_indices[:train_split]] = True
val_mask[patient_and_hs_indices[train_split:val_split]] = True
test_mask[patient_and_hs_indices[val_split:]] = True

train_data.train_mask = train_mask
train_data.val_mask = val_mask
train_data.test_mask = test_mask

# -------------------------------
# 3️⃣ Create test graph (exclude HealthScore nodes)
# -------------------------------
test_node_indices = train_data.test_mask.nonzero(as_tuple=True)[0]
test_node_list = [train_node_list[i] for i in test_node_indices]
test_node_to_idx = {n: i for i, n in enumerate(test_node_list)}

# Select edges where both nodes are in the test set
test_edges = [
    (test_node_to_idx[u], test_node_to_idx[v])
    for u, v in G.edges()
    if u in test_node_to_idx and v in test_node_to_idx
]
test_edge_index = torch.tensor(test_edges, dtype=torch.long).t().contiguous()

# Node features and labels for the test graph
test_node_features = []
test_y = []

for n in test_node_list:
    data_n = G.nodes[n]
    node_type = data_n['type']
    features = [0.0]*10

    if node_type == "Patient":
        features[0] = 1
        features[1:4] = [
            safe_float(data_n.get('age')),
            safe_float(data_n.get('weight')),
            safe_float(data_n.get('height'))
        ]
        features[4] = encode_smoker(data_n.get('smoker'))
        features[5] = encode_alcohol(data_n.get('alcohol_consumption'))
        test_y.append(safe_float(data_n.get("Health_Score", 0)))

    elif node_type == "Measurement":
        # Encode Normal vs Abnormal
        if n.startswith("Normal"):
            features[8] = 1
            features[9] = 0
        elif n.startswith("Abnormal"):
            features[8] = 0
            features[9] = 1
        else:
            features[8] = 0
            features[9] = 0
        test_y.append(-1)  # measurement nodes have no target

    elif node_type == "HealthScore":
        features[7] = 1
        test_y.append(data_n.get('value', 0))

    test_node_features.append(features)

test_x = torch.tensor(test_node_features, dtype=torch.float)
test_y = torch.tensor(test_y, dtype=torch.float)
test_data = Data(x=test_x, edge_index=test_edge_index, y=test_y)

# -------------------------------
# Summary
# -------------------------------
print("Training graph nodes:", train_data.num_nodes, "edges:", train_data.edge_index.shape[1])
print("Test graph nodes:", test_data.num_nodes, "edges:", test_data.edge_index.shape[1])


Training graph nodes: 20016 edges: 80000
Test graph nodes: 3000 edges: 217


In [ ]:
# -------------------------------
# Graph Statistics
# -------------------------------

# Count nodes by type
node_types = [G.nodes[n]['type'] for n in G.nodes()]
num_patients = node_types.count("Patient")
num_measurements = node_types.count("Measurement")
num_healthscores = node_types.count("HealthScore")
total_nodes = len(G.nodes())
total_edges = len(G.edges())

print(f"Total nodes: {total_nodes}")
print(f" - Patient nodes: {num_patients}")
print(f" - Measurement nodes: {num_measurements}")
print(f" - HealthScore nodes: {num_healthscores}")
print(f"Total edges: {total_edges}")

# HealthScore statistics
health_scores = [
    safe_float(G.nodes[n].get('value', 0))
    for n in G.nodes()
    if G.nodes[n]['type'] == "HealthScore"
]

if health_scores:
    print(f"HealthScore stats:")
    print(f" - Minimum: {min(health_scores)}")
    print(f" - Maximum: {max(health_scores)}")
    print(f" - Mean: {sum(health_scores)/len(health_scores):.2f}")
    print(f" - Median: {sorted(health_scores)[len(health_scores)//2]}")
else:
    print("No HealthScore nodes found.")

# Optional: Basic statistics for Patient Health_Score if it exists
patient_scores = [
    safe_float(G.nodes[n].get('Health_Score', 0))
    for n in G.nodes()
    if G.nodes[n]['type'] == "Patient"
]

if patient_scores:
    print(f"Patient Health_Score stats:")
    print(f" - Minimum: {min(patient_scores)}")
    print(f" - Maximum: {max(patient_scores)}")
    print(f" - Mean: {sum(patient_scores)/len(patient_scores):.2f}")
    print(f" - Median: {sorted(patient_scores)[len(patient_scores)//2]}")
else:
    print("No Patient Health_Score found.")


Total nodes: 20016
 - Patient nodes: 10000
 - Measurement nodes: 16
 - HealthScore nodes: 10000
Total edges: 80000
HealthScore stats:
 - Minimum: 0.0
 - Maximum: 100.0
 - Mean: 49.69
 - Median: 49.57893939
Patient Health_Score stats:
 - Minimum: 0.0
 - Maximum: 0.0
 - Mean: 0.00
 - Median: 0.0


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GCNConv, GATConv, GINConv

class HealthGNN_SAGE_Study(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.dropout = dropout
        self.convs.append(SAGEConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(SAGEConv(hidden_channels, hidden_channels))
        last_dim = hidden_channels // 2 if num_layers > 1 else hidden_channels
        if num_layers > 1: self.convs.append(SAGEConv(hidden_channels, last_dim))
        self.lin = nn.Linear(last_dim, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x).view(-1)

class HealthGNN_GCN_Study(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.dropout = dropout
        self.convs.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
        last_dim = hidden_channels // 2 if num_layers > 1 else hidden_channels
        if num_layers > 1: self.convs.append(GCNConv(hidden_channels, last_dim))
        self.lin = nn.Linear(last_dim, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x).view(-1)

class HealthGNN_GAT_Study(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, num_layers=2, dropout=0.2, heads=4):
        super().__init__()
        self.convs = nn.ModuleList()
        self.dropout = dropout
        self.convs.append(GATConv(in_channels, hidden_channels // heads, heads=heads))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_channels, hidden_channels // heads, heads=heads))
        last_dim = hidden_channels // 2 if num_layers > 1 else hidden_channels
        if num_layers > 1: self.convs.append(GATConv(hidden_channels, last_dim, heads=1))
        self.lin = nn.Linear(last_dim, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x).view(-1)

class HealthGNN_GIN_Study(nn.Module):
    def __init__(self, in_channels, hidden_channels=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.convs = nn.ModuleList()
        self.dropout = dropout
        def make_mlp(in_d, out_d):
            return nn.Sequential(nn.Linear(in_d, out_d), nn.ReLU(), nn.Linear(out_d, out_d))
        self.convs.append(GINConv(make_mlp(in_channels, hidden_channels)))
        for _ in range(num_layers - 2):
            self.convs.append(GINConv(make_mlp(hidden_channels, hidden_channels)))
        last_dim = hidden_channels // 2 if num_layers > 1 else hidden_channels
        if num_layers > 1: self.convs.append(GINConv(make_mlp(hidden_channels, last_dim)))
        self.lin = nn.Linear(last_dim, 1)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        for conv in self.convs:
            x = conv(x, edge_index)
            x = F.dropout(x, p=self.dropout, training=self.training)
        return self.lin(x).view(-1)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np

def train_and_evaluate_model(model_instance, model_name, data, device, num_epochs=300, lr=0.001, patience=50):
    model = model_instance.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=20)

    best_val_loss, patience_counter = float('inf'), 0
    best_model_state = None
    history = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_pearson': []}

    for epoch in range(1, num_epochs + 1):
        model.train()
        optimizer.zero_grad()
        out = model(data)
        loss = loss_fn(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            full_out = model(data)
            val_out = full_out[data.val_mask]
            val_y = data.y[data.val_mask]
            v_loss = loss_fn(val_out, val_y).item()
            v_mae = mean_absolute_error(val_y.cpu().numpy(), val_out.cpu().numpy())
            v_r, _ = pearsonr(val_y.cpu().numpy(), val_out.cpu().numpy()) if len(val_y) > 1 else (0,0)

        history['train_loss'].append(loss.item())
        history['val_loss'].append(v_loss)
        history['val_mae'].append(v_mae)
        history['val_pearson'].append(v_r)

        scheduler.step(v_loss)
        if v_loss < best_val_loss:
            best_val_loss, best_model_state, patience_counter = v_loss, model.state_dict().copy(), 0
        else:
            patience_counter += 1

        if patience_counter >= patience: break

    if best_model_state: model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        test_out = model(data)[data.test_mask].cpu().numpy()
        test_y = data.y[data.test_mask].cpu().numpy()
        metrics = {
            'mae': mean_absolute_error(test_y, test_out),
            'rmse': np.sqrt(mean_squared_error(test_y, test_out)),
            'r2': r2_score(test_y, test_out),
            'pearson': pearsonr(test_y, test_out)[0] if len(test_y) > 1 else 0
        }
    return {'history': history, 'metrics': metrics}

In [ ]:
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
models_classes = {
    'SAGE': HealthGNN_SAGE_Study,
    'GCN': HealthGNN_GCN_Study,
    'GAT': HealthGNN_GAT_Study,
    'GIN': HealthGNN_GIN_Study
}

# Hyperparameter grid
hidden_options = [64, 128]
lr_options = [0.001, 0.0005]

study_results = []

# Use the train_data graph you prepared earlier
data = train_data  # <--- assign your PyG graph here

for name, m_class in models_classes.items():
    for h in hidden_options:
        for lr in lr_options:
            run_name = f"{name}_h{h}_lr{lr}"
            print(f"🔎 Testing: {run_name}...")

            # Instantiate model
            model_inst = m_class(in_channels=data.num_node_features, hidden_channels=h)

            # Train and evaluate
            res = train_and_evaluate_model(model_inst, run_name, data, device, lr=lr)

            # Save results
            study_results.append({
                'Model': name, 'Hidden': h, 'LR': lr,
                'MAE': res['metrics']['mae'],
                'RMSE': res['metrics']['rmse'],
                'Pearson': res['metrics']['pearson'],
                'R2': res['metrics']['r2'],
                'history': res['history']
            })

df_final = pd.DataFrame(study_results)
print(df_final)


🔎 Testing: SAGE_h64_lr0.001...
🔎 Testing: SAGE_h64_lr0.0005...
🔎 Testing: SAGE_h128_lr0.001...
🔎 Testing: SAGE_h128_lr0.0005...
🔎 Testing: GCN_h64_lr0.001...
🔎 Testing: GCN_h64_lr0.0005...
🔎 Testing: GCN_h128_lr0.001...
🔎 Testing: GCN_h128_lr0.0005...
🔎 Testing: GAT_h64_lr0.001...
🔎 Testing: GAT_h64_lr0.0005...
🔎 Testing: GAT_h128_lr0.001...
🔎 Testing: GAT_h128_lr0.0005...
🔎 Testing: GIN_h64_lr0.001...
🔎 Testing: GIN_h64_lr0.0005...
🔎 Testing: GIN_h128_lr0.001...
🔎 Testing: GIN_h128_lr0.0005...
   Model  Hidden      LR        MAE       RMSE   Pearson        R2  \
0   SAGE      64  0.0010   7.627695  12.955564  0.882479  0.778160   
1   SAGE      64  0.0005   7.645062  12.918249  0.882894  0.779436   
2   SAGE     128  0.0010   7.664145  12.947730  0.882454  0.778428   
3   SAGE     128  0.0005   7.680871  12.976596  0.882274  0.777439   
4    GCN      64  0.0010  24.812010  27.192138  0.237787  0.022732   
5    GCN      64  0.0005  24.681080  27.048197  0.257731  0.033051   
6    GCN  

In [ ]:
# After creating df_final
df_final = pd.DataFrame(study_results)
print(df_final)

# Save results to Excel
excel_path = "gnn_hyperparameter_study_results.xlsx"
df_final.to_excel(excel_path, index=False)
print(f"✅ Results saved to {excel_path}")

   Model  Hidden      LR        MAE       RMSE   Pearson        R2  \
0   SAGE      64  0.0010   7.462876  12.810046  0.891961  0.792384   
1   SAGE      64  0.0005   7.524120  12.842444  0.891596  0.791333   
2   SAGE     128  0.0010   7.549858  12.828513  0.891023  0.791785   
3   SAGE     128  0.0005   7.456310  12.854689  0.891388  0.790935   
4    GCN      64  0.0010  25.263021  27.595232  0.256955  0.036554   
5    GCN      64  0.0005  25.275888  27.579373  0.273522  0.037661   
6    GCN     128  0.0010  25.457666  27.746438  0.257672  0.025967   
7    GCN     128  0.0005  25.309629  27.612163  0.265250  0.035372   
8    GAT      64  0.0010  25.514685  28.351767 -0.004684 -0.016997   
9    GAT      64  0.0005   7.953034  13.134664  0.890351  0.781728   
10   GAT     128  0.0010  25.641487  28.209242  0.007282 -0.006797   
11   GAT     128  0.0005   7.801910  12.842511  0.891716  0.791330   
12   GIN      64  0.0010  18.791403  20.638124  0.857924  0.461111   
13   GIN      64  0.